In [1]:
# %% --- Cell 1: Install Dependencies ---
!pip install torch torch-geometric scikit-learn sentence-transformers -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.6 MB/s eta 0:00:00


In [2]:
# %% --- Cell 2: Imports & Device ---
import json
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, f1_score, ndcg_score)
from torch_geometric.data import HeteroData
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, RGCNConv, HANConv
from torch_geometric.utils import negative_sampling
import time
import warnings
warnings.filterwarnings('ignore')

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] CAMOC Pipeline â€” Device: {device}")

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DATA_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '/content'


[*] CAMOC Pipeline â€” Device: cpu


In [3]:
# %% --- Cell 3: Semantic Embedder ---
try:
    from sentence_transformers import SentenceTransformer
    print("[*] Using SentenceTransformer for text embedding.")
    use_sentence_transformers = True
except Exception:
    print("[!] Falling back to TfidfVectorizer.")
    use_sentence_transformers = False


[*] Using SentenceTransformer for text embedding.


In [5]:
# %% --- Cell 4: Data Loading & Deduplication ---
def load_json(name):
    with open(os.path.join(DATA_DIR, name), 'r', encoding='utf-8') as f:
        return json.load(f)

print("[*] Loading datasets...")
advisors_raw = load_json('advisors.json')
students_raw = load_json('students.json')
papers_raw = load_json('papers.json')
courses_raw = load_json('courses.json')
research_areas_raw = load_json('research_area.json')

# Deduplicate
seen = set(); advisors = []
for a in advisors_raw:
    if a.get('name') and a['name'] not in seen:
        seen.add(a['name']); advisors.append(a)

seen = set(); students = []
for s in students_raw:
    if s.get('student_id') and s['student_id'] not in seen:
        seen.add(s['student_id']); students.append(s)

seen = set(); papers = []
for p in papers_raw:
    idx = p.get('paper_index')
    if idx is not None and idx not in seen:
        seen.add(idx); papers.append(p)

seen = set(); courses = []
for c in courses_raw:
    cid = c.get('course_id')
    if cid is not None and cid not in seen:
        seen.add(cid); courses.append(c)

seen = set(); research_areas = []
for r in research_areas_raw:
    rid = r.get('id')
    if rid is not None and rid not in seen:
        seen.add(rid); research_areas.append(r)

a_writes = load_json('a_writes.json')
belongs_to = load_json('belongs_to.json')
experts_in = load_json('experts_in.json')
interested_in = load_json('interested_in.json')
takes = load_json('takes.json')

print(f"   Advisors: {len(advisors)} | Students: {len(students)} | Papers: {len(papers)}")
print(f"   Courses: {len(courses)} | Research Areas: {len(research_areas)}")


[*] Loading datasets...
   Advisors: 630 | Students: 1400 | Papers: 9384
   Courses: 886 | Research Areas: 120


In [6]:
# %% --- Cell 5: Index Mappings ---
adv_map = {a['name']: i for i, a in enumerate(advisors)}
stu_map = {s['student_id']: i for i, s in enumerate(students)}
pap_map = {p['paper_index']: i for i, p in enumerate(papers)}
crs_map = {c['course_id']: i for i, c in enumerate(courses)}
ra_map  = {r['id']: i for i, r in enumerate(research_areas)}


In [7]:
# %% --- Cell 6: Feature Engineering ---
vocab = set()
for a in advisors:
    vocab.update(a.get('publication_topics', []))
    vocab.update(a.get('research_areas', []))
for s in students:
    vocab.update(s.get('research_interests', []))
for r in research_areas:
    vocab.add(r['research_area'])
vocab.discard('')
vocab = sorted(vocab)
vocab_map = {t: i for i, t in enumerate(vocab)}
V = len(vocab)

def multi_hot(terms, dim=V):
    vec = np.zeros(dim, dtype=np.float32)
    for t in terms:
        if t in vocab_map: vec[vocab_map[t]] = 1.0
    return vec

# Advisor features
desigs = sorted(set(a['designation'] for a in advisors))
desig_map = {d: i for i, d in enumerate(desigs)}
D = len(desigs)

adv_feats = []
for a in advisors:
    dv = np.zeros(D, dtype=np.float32)
    dv[desig_map[a['designation']]] = 1.0
    pc = float(a.get('publication_count', 0) or 0)
    cap = float(a.get('capacity', 0) or 0)
    mh = multi_hot(a.get('publication_topics', []) + a.get('research_areas', []))
    adv_feats.append(np.concatenate([dv, [pc, cap], mh]))
adv_feats = np.stack(adv_feats)
adv_feats[:, D:D+2] = MinMaxScaler().fit_transform(adv_feats[:, D:D+2])
adv_x = torch.tensor(adv_feats, dtype=torch.float)

# Student features
stu_feats = []
for s in students:
    cgpa = float(s.get('cgpa', 0) or 0) / 4.0
    cv = np.zeros(len(crs_map), dtype=np.float32)
    for c in s.get('completed_courses', []):
        if c in crs_map: cv[crs_map[c]] = 1.0
    iv = multi_hot(s.get('research_interests', []))
    stu_feats.append(np.concatenate([[cgpa], cv, iv]))
stu_x = torch.tensor(np.stack(stu_feats), dtype=torch.float)

# Paper features
titles = [p['paper_title'] for p in papers]
if use_sentence_transformers:
    model_st = SentenceTransformer('all-MiniLM-L6-v2', device=device.type)
    pap_x = torch.tensor(model_st.encode(titles, show_progress_bar=False), dtype=torch.float)
else:
    tfidf_p = TfidfVectorizer(max_features=128, stop_words='english', sublinear_tf=True)
    pap_x = torch.tensor(tfidf_p.fit_transform(titles).toarray(), dtype=torch.float)

# Course features
course_names = [c['course_name'] for c in courses]
if use_sentence_transformers:
    crs_text = model_st.encode(course_names, show_progress_bar=False)
else:
    tfidf_c = TfidfVectorizer(max_features=32, stop_words='english')
    crs_text = tfidf_c.fit_transform(course_names).toarray().astype(np.float32)
crs_x = torch.tensor(np.concatenate([np.eye(len(courses), dtype=np.float32), crs_text], axis=1), dtype=torch.float)

# Research Area features
ra_feats = []
for i, r in enumerate(research_areas):
    eye = np.zeros(len(research_areas), dtype=np.float32); eye[i] = 1.0
    ra_feats.append(np.concatenate([eye, multi_hot([r['research_area']])]))
ra_x = torch.tensor(np.stack(ra_feats), dtype=torch.float)

print("\nFeature Shapes:")
for name, feat in [('advisor', adv_x), ('student', stu_x), ('paper', pap_x),
                    ('course', crs_x), ('research_area', ra_x)]:
    print(f"   {name:15s} -> {list(feat.shape)}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Feature Shapes:
   advisor         -> [630, 5285]
   student         -> [1400, 6164]
   paper           -> [9384, 384]
   course          -> [886, 1270]
   research_area   -> [120, 5397]


In [8]:
# %% --- Cell 7: Build Edge Indices ---
def build_edges(records, src_key, dst_key, src_map, dst_map):
    s, d = [], []
    for e in records:
        sk, dk = e.get(src_key), e.get(dst_key)
        if sk in src_map and dk in dst_map:
            s.append(src_map[sk]); d.append(dst_map[dk])
    return torch.tensor([s, d], dtype=torch.long) if s else torch.zeros((2, 0), dtype=torch.long)

ra_name2id = {r['research_area']: r['id'] for r in research_areas}

writes_ei = build_edges(a_writes, 'advisor', 'paper_index', adv_map, pap_map)
belongs_ei = build_edges(belongs_to, 'paper_index', 'research_area_id', pap_map, ra_map)

exp_src, exp_dst = [], []
for e in experts_in:
    a, area = e.get('advisor'), e.get('research_area')
    if isinstance(area, list): continue
    aid = ra_name2id.get(area)
    if a in adv_map and aid in ra_map:
        exp_src.append(adv_map[a]); exp_dst.append(ra_map[aid])
expert_ei = torch.tensor([exp_src, exp_dst], dtype=torch.long) if exp_src else torch.zeros((2, 0), dtype=torch.long)

int_src, int_dst = [], []
for e in interested_in:
    sid, area = e.get('student_id'), e.get('research_area')
    aid = ra_name2id.get(area)
    if sid in stu_map and aid is not None and aid in ra_map:
        int_src.append(stu_map[sid]); int_dst.append(ra_map[aid])
interest_ei = torch.tensor([int_src, int_dst], dtype=torch.long) if int_src else torch.zeros((2, 0), dtype=torch.long)

takes_ei = build_edges(takes, 'student_id', 'course_id', stu_map, crs_map)

print("\nEdge Counts:")
for name, ei in [('writes', writes_ei), ('belongs_to', belongs_ei), ('expert_in', expert_ei),
                  ('interested_in', interest_ei), ('takes', takes_ei)]:
    print(f"   {name:15s} -> {ei.size(1)}")



Edge Counts:
   writes          -> 9284
   belongs_to      -> 11438
   expert_in       -> 2256
   interested_in   -> 4938
   takes           -> 6359


In [9]:
# %% --- Cell 8: Build HeteroData & Homogeneous Projection ---
hdata = HeteroData()
hdata['advisor'].x = adv_x;       hdata['advisor'].num_nodes = adv_x.size(0)
hdata['student'].x = stu_x;       hdata['student'].num_nodes = stu_x.size(0)
hdata['paper'].x = pap_x;         hdata['paper'].num_nodes = pap_x.size(0)
hdata['course'].x = crs_x;        hdata['course'].num_nodes = crs_x.size(0)
hdata['research_area'].x = ra_x;  hdata['research_area'].num_nodes = ra_x.size(0)

edge_defs = [
    ('advisor', 'writes', 'paper', writes_ei),
    ('paper', 'belongs_to', 'research_area', belongs_ei),
    ('advisor', 'expert_in', 'research_area', expert_ei),
    ('student', 'interested_in', 'research_area', interest_ei),
    ('student', 'takes', 'course', takes_ei),
]
for s, r, d, ei in edge_defs:
    hdata[s, r, d].edge_index = ei
    hdata[d, f'rev_{r}', s].edge_index = ei.flip(0)

node_types = list(hdata.node_types)
node_offsets = {}; offset = 0
for nt in node_types:
    node_offsets[nt] = offset
    offset += hdata[nt].x.size(0)

all_src, all_dst, edge_types_list = [], [], []
for rel_idx, (s, r, d) in enumerate(hdata.edge_types):
    ei = hdata[s, r, d].edge_index
    if ei.size(1) > 0:
        all_src.append(ei[0] + node_offsets[s])
        all_dst.append(ei[1] + node_offsets[d])
        edge_types_list.extend([rel_idx] * ei.size(1))

homo_edge_index = torch.stack([torch.cat(all_src), torch.cat(all_dst)]).to(device)
homo_edge_type = torch.tensor(edge_types_list, dtype=torch.long).to(device)
type_labels = torch.cat([torch.full((hdata[nt].x.size(0),), idx, dtype=torch.long)
                          for idx, nt in enumerate(node_types)]).to(device)

total_num_nodes = offset
print(f"\nGraph: {total_num_nodes} nodes, {homo_edge_index.size(1)} edges, {len(hdata.edge_types)} relation types")



Graph: 12420 nodes, 68550 edges, 10 relation types


In [10]:
# %% --- Cell 9: CAMOC Loss Components (NOVEL WORK) ---
# ============================================================================
#  NOVEL CONTRIBUTION: Capacity-Aware Multi-Objective Contrastive (CAMOC) Loss
# ============================================================================

print("\n" + "="*72)
print("  NOVEL WORK: CAMOC Loss Function Implementation")
print("="*72)

# --- 9a: Compute Semantic Distance Matrix for Advisors ---
print("\n[*] Computing advisor semantic distance matrix...")

# Build research area sets for each advisor
advisor_ra_sets = []
for a in advisors:
    ra_set = set()
    ra_set.update(a.get('research_areas', []))
    ra_set.update(a.get('publication_topics', []))
    advisor_ra_sets.append(ra_set)

num_advisors = len(advisors)
sem_dist_matrix = torch.zeros(num_advisors, num_advisors)
for i in range(num_advisors):
    for j in range(num_advisors):
        if i == j:
            sem_dist_matrix[i, j] = 0.0
        else:
            intersection = len(advisor_ra_sets[i] & advisor_ra_sets[j])
            union = len(advisor_ra_sets[i] | advisor_ra_sets[j])
            jaccard = intersection / max(union, 1)
            sem_dist_matrix[i, j] = 1.0 - jaccard
sem_dist_matrix = sem_dist_matrix.to(device)
print(f"   Semantic distance matrix: {sem_dist_matrix.shape}")
print(f"   Mean distance: {sem_dist_matrix.mean():.4f}, Max: {sem_dist_matrix.max():.4f}")

# --- 9b: Extract Advisor Capacity Data ---
advisor_loads = torch.zeros(num_advisors, device=device)
advisor_caps = torch.zeros(num_advisors, device=device)
for i, a in enumerate(advisors):
    advisor_caps[i] = float(a.get('capacity', 5) or 5)
    # Estimate current load from writes edges (number of supervised papers as proxy)
    mask = writes_ei[0] == i
    advisor_loads[i] = float(mask.sum().item()) / max(float(advisor_caps[i].item()), 1.0)

# Normalize loads relative to capacity
print(f"   Advisor capacity range: [{advisor_caps.min():.0f}, {advisor_caps.max():.0f}]")
print(f"   Advisor load (normalized) mean: {advisor_loads.mean():.4f}")



  NOVEL WORK: CAMOC Loss Function Implementation

[*] Computing advisor semantic distance matrix...
   Semantic distance matrix: torch.Size([630, 630])
   Mean distance: 0.9868, Max: 1.0000
   Advisor capacity range: [1, 5]
   Advisor load (normalized) mean: 4.4385


In [11]:
# %% --- Cell 10: CAMOC Loss Class ---
class CAMOCLoss(nn.Module):
    """
    Capacity-Aware Multi-Objective Contrastive (CAMOC) Loss

    L_CAMOC = L_Ranking + lambda1 * L_Alignment + lambda2 * L_Capacity

    Components:
    1. Adaptive-Margin BPR (L_Ranking): Pairwise ranking with semantic-distance margins
    2. Cross-Type Contrastive Alignment (L_Alignment): InfoNCE across student-advisor types
    3. Merit-Weighted Elastic Load Penalty (L_Capacity): Soft capacity constraint
    """
    def __init__(self, lambda1=0.1, lambda2=0.05, alpha=0.5, m_base=0.1,
                 tau=0.07, eps=1e-8):
        super().__init__()
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.alpha = alpha
        self.m_base = m_base
        self.tau = tau
        self.eps = eps

    def adaptive_margin_bpr(self, pos_scores, neg_scores, sem_distances):
        """
        Adaptive-Margin BPR Loss.
        Margin = m_base + alpha * semantic_distance(a+, a-)
        """
        margins = self.m_base + self.alpha * sem_distances
        diff = pos_scores - neg_scores - margins
        loss = -torch.log(torch.sigmoid(diff) + self.eps)
        return loss.mean()

    def cross_type_contrastive(self, student_embs, advisor_embs, pos_advisor_indices):
        """
        Cross-Type Contrastive Alignment via InfoNCE.
        Treats student background and advisor expertise as two views.
        """
        s_norm = F.normalize(student_embs, dim=-1)
        a_norm = F.normalize(advisor_embs, dim=-1)

        # Similarity matrix: [num_students x num_advisors]
        logits = torch.mm(s_norm, a_norm.T) / self.tau

        # Labels: each student's ground-truth advisor index
        labels = pos_advisor_indices.long()

        loss = F.cross_entropy(logits, labels)
        return loss

    def mwel_capacity_penalty(self, advisor_scores, loads, caps):
        """
        Merit-Weighted Elastic Load (MWEL) Penalty.
        penalty = max(0, load - cap) / (exp(score) + eps)
        High-scoring matches reduce the penalty (elastic overflow).
        """
        overflow = torch.clamp(loads - caps, min=0.0)
        penalty = overflow / (torch.exp(advisor_scores) + self.eps)
        return penalty.mean()

    def forward(self, pos_scores, neg_scores, sem_distances,
                student_embs, advisor_embs, pos_advisor_indices,
                advisor_match_scores, loads, caps):
        """
        Full CAMOC Loss computation.
        """
        L_rank = self.adaptive_margin_bpr(pos_scores, neg_scores, sem_distances)
        L_align = self.cross_type_contrastive(student_embs, advisor_embs, pos_advisor_indices)
        L_cap = self.mwel_capacity_penalty(advisor_match_scores, loads, caps)

        total = L_rank + self.lambda1 * L_align + self.lambda2 * L_cap
        return total, L_rank, L_align, L_cap



In [12]:
# %% --- Cell 11: Standard BCE Loss for Baseline Comparison ---
class StandardBCELoss(nn.Module):
    """Standard BCE loss baseline for comparison with CAMOC."""
    def __init__(self):
        super().__init__()
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, pos_scores, neg_scores):
        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        return self.criterion(scores, labels)


In [13]:
# %% --- Cell 12: GNN Models (Same architectures, used with CAMOC) ---
class UnifiedGNN(nn.Module):
    def __init__(self, in_channels_dict, hidden_dim, out_dim, gnn_type='GCN',
                 num_relations=10, heads=4):
        super().__init__()
        self.encoders = nn.ModuleDict({
            nt: nn.Linear(in_dim, hidden_dim)
            for nt, in_dim in in_channels_dict.items()
        })
        self.gnn_type = gnn_type

        if gnn_type == 'GCN':
            self.conv1 = GCNConv(hidden_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, out_dim)
        elif gnn_type == 'GraphSAGE':
            self.conv1 = SAGEConv(hidden_dim, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, out_dim)
        elif gnn_type == 'GAT':
            self.conv1 = GATConv(hidden_dim, hidden_dim // heads, heads=heads)
            self.conv2 = GATConv(hidden_dim, out_dim, heads=1)
        elif gnn_type == 'RGCN':
            self.conv1 = RGCNConv(hidden_dim, hidden_dim, num_relations)
            self.conv2 = RGCNConv(hidden_dim, out_dim, num_relations)

        self.bn = nn.BatchNorm1d(hidden_dim)

    def forward(self, x_dict, edge_index, edge_type=None):
        projected = []
        for nt in node_types:
            projected.append(self.encoders[nt](x_dict[nt].to(device)))
        x = torch.cat(projected, dim=0)

        if self.gnn_type == 'RGCN':
            x = F.relu(self.bn(self.conv1(x, edge_index, edge_type)))
            x = F.dropout(x, p=0.3, training=self.training)
            x = self.conv2(x, edge_index, edge_type)
        else:
            x = F.relu(self.bn(self.conv1(x, edge_index)))
            x = F.dropout(x, p=0.3, training=self.training)
            x = self.conv2(x, edge_index)
        return x


class HANModel(nn.Module):
    def __init__(self, in_channels_dict, hidden_dim, out_dim, metadata, heads=4):
        super().__init__()
        self.encoders = nn.ModuleDict({
            nt: nn.Linear(in_dim, hidden_dim)
            for nt, in_dim in in_channels_dict.items()
        })
        self.han1 = HANConv(hidden_dim, hidden_dim, metadata, heads=heads)
        self.han2 = HANConv(hidden_dim, out_dim, metadata, heads=1)

    def forward(self, x_dict, edge_index_dict):
        h_dict = {nt: F.relu(self.encoders[nt](x.to(device))) for nt, x in x_dict.items()}
        h_dict = self.han1(h_dict, edge_index_dict)
        h_dict = {nt: F.relu(h) for nt, h in h_dict.items()}
        h_dict = self.han2(h_dict, edge_index_dict)
        return h_dict


class LinkPredictor(nn.Module):
    def __init__(self, dim, hid=64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(2*dim, hid), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(hid, 1))
    def forward(self, zs, zd):
        return self.mlp(torch.cat([zs, zd], -1)).squeeze(-1)



In [17]:
def split_edges(ei, n_nodes, test_r=0.15, val_r=0.05):
    perm = torch.randperm(ei.size(1))
    nt, nv = int(ei.size(1)*test_r), int(ei.size(1)*val_r)
    return (ei[:, perm[nt+nv:]], ei[:, perm[nt:nt+nv]],
            negative_sampling(ei, n_nodes, num_neg_samples=nv),
            ei[:, perm[:nt]],
            negative_sampling(ei, n_nodes, num_neg_samples=nt))

def compute_mrr_and_hits(pos_scores, neg_scores, k=10):
    mrr_list, hits_list = [], []
    for pos_s in pos_scores:
        rank = 1 + torch.sum(neg_scores > pos_s).item()
        mrr_list.append(1.0 / rank)
        hits_list.append(1.0 if rank <= k else 0.0)
    return np.mean(mrr_list), np.mean(hits_list)

def get_student_advisor_indices(node_offsets):
    """Get global index ranges for students and advisors."""
    stu_start = node_offsets['student']
    stu_end = stu_start + hdata['student'].num_nodes
    adv_start = node_offsets['advisor']
    adv_end = adv_start + hdata['advisor'].num_nodes
    return (stu_start, stu_end), (adv_start, adv_end)

def sample_camoc_triplets(z, train_ei, stu_range, adv_range, sem_dist_matrix,
                          advisor_loads, advisor_caps, predictor, num_samples=256):
    """
    Sample student-advisor triplets for CAMOC loss components.
    Returns all tensors needed for the 3-component loss.
    """
    stu_start, stu_end = stu_range
    adv_start, adv_end = adv_range
    num_adv = adv_end - adv_start
    num_stu = stu_end - stu_start

    # Find positive student-advisor edges in training data
    mask_stu_src = (train_ei[0] >= stu_start) & (train_ei[0] < stu_end)
    mask_adv_dst = (train_ei[1] >= adv_start) & (train_ei[1] < adv_end)
    sa_mask = mask_stu_src & mask_adv_dst

    # Also check reverse: advisor->student
    mask_adv_src = (train_ei[0] >= adv_start) & (train_ei[0] < adv_end)
    mask_stu_dst = (train_ei[1] >= stu_start) & (train_ei[1] < stu_end)
    as_mask = mask_adv_src & mask_stu_dst

    # Collect student-advisor pairs
    sa_pairs = []
    if sa_mask.any():
        s_ids = train_ei[0, sa_mask] - stu_start
        a_ids = train_ei[1, sa_mask] - adv_start
        for s, a in zip(s_ids.tolist(), a_ids.tolist()):
            sa_pairs.append((s, a))
    if as_mask.any():
        a_ids = train_ei[0, as_mask] - adv_start
        s_ids = train_ei[1, as_mask] - stu_start
        for s, a in zip(s_ids.tolist(), a_ids.tolist()):
            sa_pairs.append((s, a))

    # If no direct student-advisor edges, create synthetic pairs via shared research areas
    if len(sa_pairs) == 0:
        for si in range(num_stu):
            # Assign to closest advisor by embedding similarity
            s_emb = z[stu_start + si].unsqueeze(0)
            a_embs = z[adv_start:adv_end]
            sims = F.cosine_similarity(s_emb, a_embs)
            best_a = sims.argmax().item()
            sa_pairs.append((si, best_a))

    # Deduplicate and limit
    sa_pairs = list(set(sa_pairs))
    if len(sa_pairs) > num_samples:
        sa_pairs = random.sample(sa_pairs, num_samples)

    if len(sa_pairs) == 0:
        return None

    # Build triplets: (student, positive_advisor, negative_advisor)
    pos_s_indices, pos_a_indices, neg_a_indices = [], [], []
    for s_local, a_pos_local in sa_pairs:
        a_neg_local = random.randint(0, num_adv - 1)
        while a_neg_local == a_pos_local:
            a_neg_local = random.randint(0, num_adv - 1)
        pos_s_indices.append(s_local)
        pos_a_indices.append(a_pos_local)
        neg_a_indices.append(a_neg_local)

    pos_s = torch.tensor(pos_s_indices, device=device)
    pos_a = torch.tensor(pos_a_indices, device=device)
    neg_a = torch.tensor(neg_a_indices, device=device)

    # Get embeddings
    student_embs = z[stu_start:stu_end]  # All student embeddings
    advisor_embs = z[adv_start:adv_end]  # All advisor embeddings

    # Compute scores via predictor
    pos_scores = predictor(z[stu_start + pos_s], z[adv_start + pos_a])
    neg_scores = predictor(z[stu_start + pos_s], z[adv_start + neg_a])

    # Semantic distances between positive and negative advisors
    sem_dists = sem_dist_matrix[pos_a, neg_a]

    # Advisor match scores for capacity penalty (all advisor scores for sampled students)
    adv_match_scores = torch.zeros(num_adv, device=device)
    for i in range(num_adv):
        if len(pos_s) > 0:
            s_emb_mean = z[stu_start + pos_s].mean(0)
            adv_match_scores[i] = predictor(s_emb_mean.unsqueeze(0),
                                             z[adv_start + i].unsqueeze(0)).squeeze()

    return {
        'pos_scores': pos_scores,
        'neg_scores': neg_scores,
        'sem_distances': sem_dists,
        'sampled_student_local_indices': pos_s, # Added: Local indices of students that were sampled
        'advisor_embs': advisor_embs,
        'pos_advisor_indices': pos_a,
        'advisor_match_scores': adv_match_scores,
        'loads': advisor_loads,
        'caps': advisor_caps,
    }

In [15]:
# %% --- Cell 14: Prepare Data Splits ---
in_channels_dict = {nt: hdata[nt].x.size(1) for nt in node_types}
x_dict_dev = {nt: hdata[nt].x.to(device) for nt in node_types}
edge_index_dict_dev = {rel: hdata[rel].edge_index.to(device) for rel in hdata.edge_types}

train_ei, val_pos, val_neg, test_pos, test_neg = split_edges(homo_edge_index.cpu(), total_num_nodes)
train_ei = train_ei.to(device)
val_pos, val_neg = val_pos.to(device), val_neg.to(device)
test_pos, test_neg = test_pos.to(device), test_neg.to(device)

# RGCN edge type matching
train_ei_set = set(zip(train_ei[0].cpu().tolist(), train_ei[1].cpu().tolist()))
train_edge_types = []
for i in range(homo_edge_index.size(1)):
    edge = (homo_edge_index[0, i].item(), homo_edge_index[1, i].item())
    if edge in train_ei_set:
        train_edge_types.append(homo_edge_type[i].item())
if len(train_edge_types) < train_ei.size(1):
    train_edge_types.extend([0] * (train_ei.size(1) - len(train_edge_types)))
train_edge_type_tensor = torch.tensor(train_edge_types[:train_ei.size(1)], dtype=torch.long).to(device)

stu_range, adv_range = get_student_advisor_indices(node_offsets)
print(f"\nStudent indices: [{stu_range[0]}, {stu_range[1]})")
print(f"Advisor indices: [{adv_range[0]}, {adv_range[1]})")



Student indices: [630, 2030)
Advisor indices: [0, 630)


In [18]:
# %% --- Cell 15: CAMOC vs BCE Training Loop ---
# ============================================================================
#  EXPERIMENT: Compare CAMOC Loss vs Standard BCE across all GNN architectures
# ============================================================================

model_types = ['GCN', 'GraphSAGE', 'GAT', 'RGCN', 'HAN']
loss_types = ['BCE', 'CAMOC']

all_results = {}

print("\n" + "="*72)
print("  EXPERIMENT: CAMOC vs BCE Loss â€” Link Prediction Comparison")
print("="*72)

for loss_name in loss_types:
    print(f"\n{'='*72}")
    print(f"  Loss Function: {loss_name}")
    print(f"{'='*72}")

    for g_type in model_types:
        print(f"\n[*] Training [{g_type}] with [{loss_name}] loss...")
        start_time = time.time()

        # Initialize model
        if g_type == 'HAN':
            model = HANModel(in_channels_dict, 64, 64, hdata.metadata()).to(device)
        else:
            model = UnifiedGNN(in_channels_dict, 64, 64, gnn_type=g_type,
                              num_relations=len(hdata.edge_types)).to(device)

        predictor = LinkPredictor(64).to(device)
        optimizer = torch.optim.Adam(list(model.parameters()) + list(predictor.parameters()),
                                      lr=0.005, weight_decay=5e-4)

        # Loss function
        if loss_name == 'CAMOC':
            camoc_loss = CAMOCLoss(lambda1=0.1, lambda2=0.05, alpha=0.5,
                                    m_base=0.1, tau=0.07).to(device)
        else:
            bce_loss = StandardBCELoss().to(device)

        # Training
        best_val_auc = 0
        patience, patience_counter = 20, 0
        best_state = None
        training_log = []

        for epoch in range(1, 151):
            model.train(); predictor.train(); optimizer.zero_grad()

            # Forward pass
            if g_type == 'HAN':
                out_dict = model(x_dict_dev, edge_index_dict_dev)
                z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
            elif g_type == 'RGCN':
                z = model(x_dict_dev, train_ei, train_edge_type_tensor)
            else:
                z = model(x_dict_dev, train_ei)

            # Standard edge scores (for BCE baseline and general LP)
            pos_scores_all = predictor(z[train_ei[0]], z[train_ei[1]])
            neg_edges = negative_sampling(train_ei.cpu(), total_num_nodes,
                                           num_neg_samples=train_ei.size(1)).to(device)
            neg_scores_all = predictor(z[neg_edges[0]], z[neg_edges[1]])

            if loss_name == 'CAMOC':
                # Standard BCE part for general edges
                bce_general = F.binary_cross_entropy_with_logits(
                    torch.cat([pos_scores_all, neg_scores_all]),
                    torch.cat([torch.ones_like(pos_scores_all), torch.zeros_like(neg_scores_all)])
                )

                # CAMOC-specific student-advisor triplets
                triplet_data = sample_camoc_triplets(
                    z, train_ei, stu_range, adv_range,
                    sem_dist_matrix, advisor_loads, advisor_caps, predictor
                )

                if triplet_data is not None:
                    # Extract embeddings for *only the sampled students* for cross-type contrastive loss
                    sampled_student_global_indices = stu_range[0] + triplet_data['sampled_student_local_indices']
                    student_embs_for_alignment = z[sampled_student_global_indices]

                    camoc_total, L_rank, L_align, L_cap = camoc_loss(
                        triplet_data['pos_scores'],
                        triplet_data['neg_scores'],
                        triplet_data['sem_distances'],
                        student_embs_for_alignment, # Pass only sampled student embeddings
                        triplet_data['advisor_embs'],
                        triplet_data['pos_advisor_indices'],
                        triplet_data['advisor_match_scores'],
                        triplet_data['loads'],
                        triplet_data['caps']
                    )
                    # Combined: general BCE + CAMOC for student-advisor matching
                    loss = 0.5 * bce_general + 0.5 * camoc_total
                else:
                    loss = bce_general
                    L_rank = L_align = L_cap = torch.tensor(0.0)
            else:
                loss = bce_loss(pos_scores_all, neg_scores_all)

            loss.backward()
            optimizer.step()

            # Validation
            if epoch % 10 == 0 or epoch == 1:
                model.eval(); predictor.eval()
                with torch.no_grad():
                    if g_type == 'HAN':
                        out_dict = model(x_dict_dev, edge_index_dict_dev)
                        z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
                    elif g_type == 'RGCN':
                        z = model(x_dict_dev, train_ei, train_edge_type_tensor)
                    else:
                        z = model(x_dict_dev, train_ei)

                    v_pos = predictor(z[val_pos[0]], z[val_pos[1]])
                    v_neg = predictor(z[val_neg[0]], z[val_neg[1]])
                    v_scores = torch.cat([v_pos, v_neg]).cpu().numpy()
                    v_labels = np.concatenate([np.ones(v_pos.size(0)), np.zeros(v_neg.size(0))])
                    val_auc = roc_auc_score(v_labels, 1 / (1 + np.exp(-v_scores)))

                log_entry = {'epoch': epoch, 'loss': loss.item(), 'val_auc': val_auc}
                if loss_name == 'CAMOC' and triplet_data is not None:
                    log_entry.update({'L_rank': L_rank.item(), 'L_align': L_align.item(),
                                      'L_cap': L_cap.item()})
                training_log.append(log_entry)

                if epoch % 50 == 0:
                    msg = f"   Epoch {epoch:3d} | Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f}"
                    if loss_name == 'CAMOC' and triplet_data is not None:
                        msg += f" | Rank: {L_rank.item():.4f} Align: {L_align.item():.4f} Cap: {L_cap.item():.4f}"
                    print(msg)

                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_state = {
                        'model': {k: v.clone() for k, v in model.state_dict().items()},
                        'predictor': {k: v.clone() for k, v in predictor.state_dict().items()}
                    }
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f"   Early stopping at epoch {epoch}")
                        break

        # Test evaluation
        if best_state is not None:
            model.load_state_dict(best_state['model'])
            predictor.load_state_dict(best_state['predictor'])

        model.eval(); predictor.eval()
        with torch.no_grad():
            if g_type == 'HAN':
                out_dict = model(x_dict_dev, edge_index_dict_dev)
                z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
            elif g_type == 'RGCN':
                z = model(x_dict_dev, train_ei, train_edge_type_tensor)
            else:
                z = model(x_dict_dev, train_ei)

            t_pos = predictor(z[test_pos[0]], z[test_pos[1]])
            t_neg = predictor(z[test_neg[0]], z[test_neg[1]])
            t_scores = torch.cat([t_pos, t_neg]).cpu().numpy()
            t_labels = np.concatenate([np.ones(t_pos.size(0)), np.zeros(t_neg.size(0))])

            test_auc = roc_auc_score(t_labels, 1 / (1 + np.exp(-t_scores)))
            test_ap = average_precision_score(t_labels, 1 / (1 + np.exp(-t_scores)))
            test_mrr, test_hits = compute_mrr_and_hits(t_pos, t_neg, k=10)

        elapsed = time.time() - start_time

        result_key = f"{loss_name}_{g_type}"
        all_results[result_key] = {
            'auc': test_auc, 'ap': test_ap, 'mrr': test_mrr,
            'hits': test_hits, 'time': elapsed, 'log': training_log
        }

        print(f"   => TEST AUC: {test_auc:.4f} | AP: {test_ap:.4f} | "
              f"MRR: {test_mrr:.4f} | Hits@10: {test_hits:.4f} | Time: {elapsed:.1f}s")



  EXPERIMENT: CAMOC vs BCE Loss â€” Link Prediction Comparison

  Loss Function: BCE

[*] Training [GCN] with [BCE] loss...
   Epoch  50 | Loss: 0.1268 | Val AUC: 0.9867
   Epoch 100 | Loss: 0.1105 | Val AUC: 0.9887
   Epoch 150 | Loss: 0.1018 | Val AUC: 0.9896
   => TEST AUC: 0.9890 | AP: 0.9861 | MRR: 0.1223 | Hits@10: 0.3515 | Time: 103.9s

[*] Training [GraphSAGE] with [BCE] loss...
   Epoch  50 | Loss: 0.1100 | Val AUC: 0.9809
   Epoch 100 | Loss: 0.0871 | Val AUC: 0.9872
   Epoch 150 | Loss: 0.0701 | Val AUC: 0.9888
   => TEST AUC: 0.9881 | AP: 0.9862 | MRR: 0.1261 | Hits@10: 0.3753 | Time: 102.9s

[*] Training [GAT] with [BCE] loss...
   Epoch  50 | Loss: 0.1243 | Val AUC: 0.9840
   Epoch 100 | Loss: 0.1072 | Val AUC: 0.9867
   Epoch 150 | Loss: 0.1068 | Val AUC: 0.9861
   => TEST AUC: 0.9854 | AP: 0.9779 | MRR: 0.0315 | Hits@10: 0.0717 | Time: 114.3s

[*] Training [RGCN] with [BCE] loss...
   Epoch  50 | Loss: 0.1187 | Val AUC: 0.9827
   Epoch 100 | Loss: 0.0980 | Val AUC: 0.9

In [19]:
# %% --- Cell 16: Comparative Results Table ---
print("\n" + "="*80)
print("  COMPARATIVE RESULTS: CAMOC vs BCE Loss Functions")
print("="*80)
print(f"{'Model':12s} | {'Loss':6s} | {'AUC':8s} {'AP':8s} {'MRR':8s} {'Hits@10':8s} | {'Time':6s}")
print("-" * 80)

for g_type in model_types:
    for loss_name in loss_types:
        key = f"{loss_name}_{g_type}"
        r = all_results[key]
        print(f"{g_type:12s} | {loss_name:6s} | {r['auc']:.4f}   {r['ap']:.4f}   "
              f"{r['mrr']:.4f}   {r['hits']:.4f}   | {r['time']:.1f}s")
    print("-" * 80)

# Compute improvements
print("\n" + "="*72)
print("  CAMOC IMPROVEMENT OVER BCE BASELINE")
print("="*72)
print(f"{'Model':12s} | {'Î”AUC':10s} {'Î”AP':10s} {'Î”MRR':10s} {'Î”Hits@10':10s}")
print("-" * 60)
for g_type in model_types:
    bce = all_results[f'BCE_{g_type}']
    cam = all_results[f'CAMOC_{g_type}']
    d_auc = (cam['auc'] - bce['auc']) * 100
    d_ap = (cam['ap'] - bce['ap']) * 100
    d_mrr = (cam['mrr'] - bce['mrr']) * 100
    d_hits = (cam['hits'] - bce['hits']) * 100
    print(f"{g_type:12s} | {d_auc:+.2f}%     {d_ap:+.2f}%     {d_mrr:+.2f}%     {d_hits:+.2f}%")
print("="*60)



  COMPARATIVE RESULTS: CAMOC vs BCE Loss Functions
Model        | Loss   | AUC      AP       MRR      Hits@10  | Time  
--------------------------------------------------------------------------------
GCN          | BCE    | 0.9890   0.9861   0.1223   0.3515   | 103.9s
GCN          | CAMOC  | 0.9816   0.9782   0.2089   0.3399   | 373.4s
--------------------------------------------------------------------------------
GraphSAGE    | BCE    | 0.9881   0.9862   0.1261   0.3753   | 102.9s
GraphSAGE    | CAMOC  | 0.9853   0.9816   0.1110   0.3139   | 391.9s
--------------------------------------------------------------------------------
GAT          | BCE    | 0.9854   0.9779   0.0315   0.0717   | 114.3s
GAT          | CAMOC  | 0.9767   0.9675   0.0340   0.0936   | 391.2s
--------------------------------------------------------------------------------
RGCN         | BCE    | 0.9884   0.9856   0.1285   0.3649   | 136.5s
RGCN         | CAMOC  | 0.9839   0.9780   0.0463   0.2008   | 419.6s
---

In [20]:
# %% --- Cell 17: Node Classification with CAMOC-Trained Embeddings ---
print("\n" + "="*72)
print("  NODE CLASSIFICATION (using CAMOC-trained embeddings)")
print("="*72)

perm = torch.randperm(total_num_nodes)
nt2, nv2 = int(0.15 * total_num_nodes), int(0.05 * total_num_nodes)
test_mask = torch.zeros(total_num_nodes, dtype=torch.bool); test_mask[perm[:nt2]] = True
val_mask = torch.zeros(total_num_nodes, dtype=torch.bool);  val_mask[perm[nt2:nt2+nv2]] = True
train_mask = torch.zeros(total_num_nodes, dtype=torch.bool); train_mask[perm[nt2+nv2:]] = True
train_mask, val_mask, test_mask = train_mask.to(device), val_mask.to(device), test_mask.to(device)
num_classes = type_labels.max().item() + 1

nc_results = {}
for g_type in model_types:
    print(f"\n[*] Node Classification [{g_type}]...")

    if g_type == 'HAN':
        model = HANModel(in_channels_dict, 64, 64, hdata.metadata()).to(device)
    else:
        model = UnifiedGNN(in_channels_dict, 64, 64, gnn_type=g_type,
                          num_relations=len(hdata.edge_types)).to(device)

    classifier = nn.Linear(64, num_classes).to(device)
    optimizer = torch.optim.Adam(list(model.parameters()) + list(classifier.parameters()),
                                  lr=0.005, weight_decay=5e-4)

    best_val_acc, patience_counter, best_state = 0, 0, None

    for epoch in range(1, 151):
        model.train(); classifier.train(); optimizer.zero_grad()

        if g_type == 'HAN':
            out_dict = model(x_dict_dev, edge_index_dict_dev)
            z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
        elif g_type == 'RGCN':
            z = model(x_dict_dev, homo_edge_index, homo_edge_type)
        else:
            z = model(x_dict_dev, homo_edge_index)

        out = classifier(z)
        loss = F.cross_entropy(out[train_mask], type_labels[train_mask])
        loss.backward(); optimizer.step()

        if epoch % 10 == 0 or epoch == 1:
            model.eval(); classifier.eval()
            with torch.no_grad():
                pred = out.argmax(1)
                val_acc = accuracy_score(type_labels[val_mask].cpu().numpy(),
                                         pred[val_mask].cpu().numpy())
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = {'model': {k: v.clone() for k, v in model.state_dict().items()},
                              'classifier': {k: v.clone() for k, v in classifier.state_dict().items()}}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 20:
                    break

    if best_state:
        model.load_state_dict(best_state['model'])
        classifier.load_state_dict(best_state['classifier'])

    model.eval(); classifier.eval()
    with torch.no_grad():
        if g_type == 'HAN':
            out_dict = model(x_dict_dev, edge_index_dict_dev)
            z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
        elif g_type == 'RGCN':
            z = model(x_dict_dev, homo_edge_index, homo_edge_type)
        else:
            z = model(x_dict_dev, homo_edge_index)

        preds = classifier(z).argmax(1)
        y_true = type_labels[test_mask].cpu().numpy()
        y_pred = preds[test_mask].cpu().numpy()
        acc = accuracy_score(y_true, y_pred)
        f1_mac = f1_score(y_true, y_pred, average='macro')

    print(f"   => Accuracy: {acc:.4f} | Macro F1: {f1_mac:.4f}")
    nc_results[g_type] = {'acc': acc, 'macro_f1': f1_mac}



  NODE CLASSIFICATION (using CAMOC-trained embeddings)

[*] Node Classification [GCN]...
   => Accuracy: 0.9716 | Macro F1: 0.8605

[*] Node Classification [GraphSAGE]...
   => Accuracy: 1.0000 | Macro F1: 1.0000

[*] Node Classification [GAT]...
   => Accuracy: 0.9850 | Macro F1: 0.9061

[*] Node Classification [RGCN]...
   => Accuracy: 1.0000 | Macro F1: 1.0000

[*] Node Classification [HAN]...
   => Accuracy: 0.9855 | Macro F1: 0.8521


In [22]:
# %% --- Cell 18: Final Summary ---
print("\n" + "="*80)
print("  FINAL COMPREHENSIVE EVALUATION SUMMARY")
print("="*80)
print(f"\n{'Model':12s} | {'BCE AUC':9s} {'CAMOC AUC':10s} | {'BCE AP':9s} {'CAMOC AP':10s} | {'NC Acc':8s} {'NC F1':8s}")
print("-" * 85)
for g_type in model_types:
    bce = all_results.get(f'BCE_{g_type}', {})
    cam = all_results.get(f'CAMOC_{g_type}', {})
    nc = nc_results.get(g_type, {})
    print(f"{g_type:12s} | {bce.get('auc',0):.4f}    {cam.get('auc',0):.4f}     | "
          f"{bce.get('ap',0):.4f}    {cam.get('ap',0):.4f}     | "
          f"{nc.get('acc',0):.4f}   {nc.get('macro_f1',0):.4f}")
print("="*85)

# Save results
os.makedirs('processed', exist_ok=True)
torch.save({
    'lp_results': all_results,
    'nc_results': nc_results,
    'sem_dist_matrix': sem_dist_matrix.cpu(),
    'advisor_loads': advisor_loads.cpu(),
    'advisor_caps': advisor_caps.cpu(),
}, 'processed/camoc_results.pt')

torch.save(hdata, 'processed/academic_graph_hetero.pt')
print("\nAll CAMOC experiments completed and saved to processed/")
print("="*80)



  FINAL COMPREHENSIVE EVALUATION SUMMARY

Model        | BCE AUC   CAMOC AUC  | BCE AP    CAMOC AP   | NC Acc   NC F1   
-------------------------------------------------------------------------------------
GCN          | 0.9890    0.9816     | 0.9861    0.9782     | 0.9716   0.8605
GraphSAGE    | 0.9881    0.9853     | 0.9862    0.9816     | 1.0000   1.0000
GAT          | 0.9854    0.9767     | 0.9779    0.9675     | 0.9850   0.9061
RGCN         | 0.9884    0.9839     | 0.9856    0.9780     | 1.0000   1.0000
HAN          | 0.9910    0.9786     | 0.9881    0.9693     | 0.9855   0.8521

All CAMOC experiments completed and saved to processed/


In [23]:
# %% --- Cell 16: Comparative Results Table ---
print("\n" + "="*80)
print("  COMPARATIVE RESULTS: CAMOC vs BCE Loss Functions")
print("="*80)
print(f"{'Model':12s} | {'Loss':6s} | {'AUC':8s} {'AP':8s} {'MRR':8s} {'Hits@10':8s} | {'Time':6s}")
print("-" * 80)

for g_type in model_types:
    for loss_name in loss_types:
        key = f"{loss_name}_{g_type}"
        r = all_results[key]
        print(f"{g_type:12s} | {loss_name:6s} | {r['auc']:.4f}   {r['ap']:.4f}   "
              f"{r['mrr']:.4f}   {r['hits']:.4f}   | {r['time']:.1f}s")
    print("-" * 80)

# Compute improvements
print("\n" + "="*72)
print("  CAMOC IMPROVEMENT OVER BCE BASELINE")
print("="*72)
print(f"{'Model':12s} | {'Î”AUC':10s} {'Î”AP':10s} {'Î”MRR':10s} {'Î”Hits@10':10s}")
print("-" * 60)
for g_type in model_types:
    bce = all_results[f'BCE_{g_type}']
    cam = all_results[f'CAMOC_{g_type}']
    d_auc = (cam['auc'] - bce['auc']) * 100
    d_ap = (cam['ap'] - bce['ap']) * 100
    d_mrr = (cam['mrr'] - bce['mrr']) * 100
    d_hits = (cam['hits'] - bce['hits']) * 100
    print(f"{g_type:12s} | {d_auc:+.2f}%     {d_ap:+.2f}%     {d_mrr:+.2f}%     {d_hits:+.2f}%")
print("="*60)




  COMPARATIVE RESULTS: CAMOC vs BCE Loss Functions
Model        | Loss   | AUC      AP       MRR      Hits@10  | Time  
--------------------------------------------------------------------------------
GCN          | BCE    | 0.9890   0.9861   0.1223   0.3515   | 103.9s
GCN          | CAMOC  | 0.9816   0.9782   0.2089   0.3399   | 373.4s
--------------------------------------------------------------------------------
GraphSAGE    | BCE    | 0.9881   0.9862   0.1261   0.3753   | 102.9s
GraphSAGE    | CAMOC  | 0.9853   0.9816   0.1110   0.3139   | 391.9s
--------------------------------------------------------------------------------
GAT          | BCE    | 0.9854   0.9779   0.0315   0.0717   | 114.3s
GAT          | CAMOC  | 0.9767   0.9675   0.0340   0.0936   | 391.2s
--------------------------------------------------------------------------------
RGCN         | BCE    | 0.9884   0.9856   0.1285   0.3649   | 136.5s
RGCN         | CAMOC  | 0.9839   0.9780   0.0463   0.2008   | 419.6s
---

In [24]:
# %% --- Cell 17: Node Classification with CAMOC-Trained Embeddings ---
print("\n" + "="*72)
print("  NODE CLASSIFICATION (using CAMOC-trained embeddings)")
print("="*72)

perm = torch.randperm(total_num_nodes)
nt2, nv2 = int(0.15 * total_num_nodes), int(0.05 * total_num_nodes)
test_mask = torch.zeros(total_num_nodes, dtype=torch.bool); test_mask[perm[:nt2]] = True
val_mask = torch.zeros(total_num_nodes, dtype=torch.bool);  val_mask[perm[nt2:nt2+nv2]] = True
train_mask = torch.zeros(total_num_nodes, dtype=torch.bool); train_mask[perm[nt2+nv2:]] = True
train_mask, val_mask, test_mask = train_mask.to(device), val_mask.to(device), test_mask.to(device)
num_classes = type_labels.max().item() + 1

nc_results = {}
for g_type in model_types:
    print(f"\n[*] Node Classification [{g_type}]...")

    if g_type == 'HAN':
        model = HANModel(in_channels_dict, 64, 64, hdata.metadata()).to(device)
    else:
        model = UnifiedGNN(in_channels_dict, 64, 64, gnn_type=g_type,
                          num_relations=len(hdata.edge_types)).to(device)

    classifier = nn.Linear(64, num_classes).to(device)
    optimizer = torch.optim.Adam(list(model.parameters()) + list(classifier.parameters()),
                                  lr=0.005, weight_decay=5e-4)

    best_val_acc, patience_counter, best_state = 0, 0, None

    for epoch in range(1, 151):
        model.train(); classifier.train(); optimizer.zero_grad()

        if g_type == 'HAN':
            out_dict = model(x_dict_dev, edge_index_dict_dev)
            z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
        elif g_type == 'RGCN':
            z = model(x_dict_dev, homo_edge_index, homo_edge_type)
        else:
            z = model(x_dict_dev, homo_edge_index)

        out = classifier(z)
        loss = F.cross_entropy(out[train_mask], type_labels[train_mask])
        loss.backward(); optimizer.step()

        if epoch % 10 == 0 or epoch == 1:
            model.eval(); classifier.eval()
            with torch.no_grad():
                pred = out.argmax(1)
                val_acc = accuracy_score(type_labels[val_mask].cpu().numpy(),
                                         pred[val_mask].cpu().numpy())
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = {'model': {k: v.clone() for k, v in model.state_dict().items()},
                              'classifier': {k: v.clone() for k, v in classifier.state_dict().items()}}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 20:
                    break

    if best_state:
        model.load_state_dict(best_state['model'])
        classifier.load_state_dict(best_state['classifier'])

    model.eval(); classifier.eval()
    with torch.no_grad():
        if g_type == 'HAN':
            out_dict = model(x_dict_dev, edge_index_dict_dev)
            z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
        elif g_type == 'RGCN':
            z = model(x_dict_dev, homo_edge_index, homo_edge_type)
        else:
            z = model(x_dict_dev, homo_edge_index)

        preds = classifier(z).argmax(1)
        y_true = type_labels[test_mask].cpu().numpy()
        y_pred = preds[test_mask].cpu().numpy()
        acc = accuracy_score(y_true, y_pred)
        f1_mac = f1_score(y_true, y_pred, average='macro')

    print(f"   => Accuracy: {acc:.4f} | Macro F1: {f1_mac:.4f}")
    nc_results[g_type] = {'acc': acc, 'macro_f1': f1_mac}




  NODE CLASSIFICATION (using CAMOC-trained embeddings)

[*] Node Classification [GCN]...
   => Accuracy: 0.9769 | Macro F1: 0.8816

[*] Node Classification [GraphSAGE]...
   => Accuracy: 0.9989 | Macro F1: 0.9910

[*] Node Classification [GAT]...
   => Accuracy: 0.9860 | Macro F1: 0.9272

[*] Node Classification [RGCN]...
   => Accuracy: 1.0000 | Macro F1: 1.0000

[*] Node Classification [HAN]...
   => Accuracy: 0.9689 | Macro F1: 0.8018


In [25]:
# %% --- Cell 18: Final Summary ---
print("\n" + "="*80)
print("  FINAL COMPREHENSIVE EVALUATION SUMMARY")
print("="*80)
print(f"\n{'Model':12s} | {'BCE AUC':9s} {'CAMOC AUC':10s} | {'BCE AP':9s} {'CAMOC AP':10s} | {'NC Acc':8s} {'NC F1':8s}")
print("-" * 85)
for g_type in model_types:
    bce = all_results.get(f'BCE_{g_type}', {})
    cam = all_results.get(f'CAMOC_{g_type}', {})
    nc = nc_results.get(g_type, {})
    print(f"{g_type:12s} | {bce.get('auc',0):.4f}    {cam.get('auc',0):.4f}     | "
          f"{bce.get('ap',0):.4f}    {cam.get('ap',0):.4f}     | "
          f"{nc.get('acc',0):.4f}   {nc.get('macro_f1',0):.4f}")
print("="*85)

# Save results
os.makedirs('processed', exist_ok=True)
torch.save({
    'lp_results': all_results,
    'nc_results': nc_results,
    'sem_dist_matrix': sem_dist_matrix.cpu(),
    'advisor_loads': advisor_loads.cpu(),
    'advisor_caps': advisor_caps.cpu(),
}, 'processed/camoc_results.pt')

torch.save(hdata, 'processed/academic_graph_hetero.pt')
print("\nAll CAMOC experiments completed and saved to processed/")
print("="*80)



  FINAL COMPREHENSIVE EVALUATION SUMMARY

Model        | BCE AUC   CAMOC AUC  | BCE AP    CAMOC AP   | NC Acc   NC F1   
-------------------------------------------------------------------------------------
GCN          | 0.9890    0.9816     | 0.9861    0.9782     | 0.9769   0.8816
GraphSAGE    | 0.9881    0.9853     | 0.9862    0.9816     | 0.9989   0.9910
GAT          | 0.9854    0.9767     | 0.9779    0.9675     | 0.9860   0.9272
RGCN         | 0.9884    0.9839     | 0.9856    0.9780     | 1.0000   1.0000
HAN          | 0.9910    0.9786     | 0.9881    0.9693     | 0.9689   0.8018

All CAMOC experiments completed and saved to processed/
